In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import entropy, skew, kurtosis
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.filters import sobel
from skimage.util import img_as_ubyte

In [3]:
!{sys.executable} -m pip install scikit-image


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\jessi\myenv\Scripts\python.exe -m pip install --upgrade pip


In [4]:
DATASET_PATH = Path(r"C:\Users\jessi\OneDrive\Desktop\aiims2")
manifest     = pd.read_csv(DATASET_PATH / 'processed_manifest.csv')

print(f"Total cases: {len(manifest)}")
manifest.head()

Total cases: 780


,case_id,label,npz_path,orig_mean,orig_std,tumor_pixels,ring5_pixels,ring15_pixels,posterior_pixels,skin_pixels,bg_pixels
0,benign (1),benign,C:\Users\jessi\OneDrive\Desktop\aiims2\process...,132.3680,57.4818,246,325,950,43264,7680,64015
1,benign (10),benign,C:\Users\jessi\OneDrive\Desktop\aiims2\process...,103.2397,57.8553,5406,1283,2673,35840,7680,56174
2,benign (100),benign,C:\Users\jessi\OneDrive\Desktop\aiims2\process...,90.1100,67.4156,3040,1023,2330,34816,7680,59143
3,benign (101),benign,C:\Users\jessi\OneDrive\Desktop\aiims2\process...,103.2252,56.6992,1032,606,1481,34560,7680,62417
4,benign (102),benign,C:\Users\jessi\OneDrive\Desktop\aiims2\process...,54.3837,36.9740,3199,994,2280,46336,7680,59063


In [5]:
def extract_region_features(image, region_mask):
    """
    Given a normalized image and a binary region mask,
    extracts all handcrafted texture features for that region.
    Returns a flat dictionary of features.
    """
    features = {}

    # Get pixel values in this region
    pixels = image[region_mask == 1]

    # Skip empty regions
    if len(pixels) < 10:
        return None

    # --- Basic intensity stats ---
    features['mean_intensity'] = float(np.mean(pixels))
    features['std_intensity']  = float(np.std(pixels))
    features['skewness']       = float(skew(pixels))
    features['kurtosis']       = float(kurtosis(pixels))

    # --- Entropy ---
    # need histogram of pixel values for entropy
    hist, _ = np.histogram(pixels, bins=256, density=True)
    hist    = hist[hist > 0]   # remove zeros before log
    features['entropy'] = float(entropy(hist))

    # --- GLCM features ---
    # GLCM needs uint8 image
    # clip normalized image back to 0-255 uint8 for GLCM
    img_clipped = np.clip(image, image.min(), image.max())
    img_uint8   = ((img_clipped - img_clipped.min()) /
                   (img_clipped.max() - img_clipped.min()) * 255).astype(np.uint8)

    # Apply mask — set non-region pixels to 0
    img_masked = img_uint8.copy()
    img_masked[region_mask == 0] = 0

    # Reduce to 64 gray levels for GLCM (faster, less sparse)
    img_64 = (img_masked // 4).astype(np.uint8)

    glcm = graycomatrix(
        img_64,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=64,
        symmetric=True,
        normed=True
    )

    # Average across all angles
    features['glcm_contrast']    = float(graycoprops(glcm, 'contrast').mean())
    features['glcm_homogeneity'] = float(graycoprops(glcm, 'homogeneity').mean())
    features['glcm_energy']      = float(graycoprops(glcm, 'energy').mean())
    features['glcm_correlation'] = float(graycoprops(glcm, 'correlation').mean())

    # --- LBP histogram features ---
    # LBP on full image, then extract values within region
    lbp = local_binary_pattern(img_uint8, P=8, R=1, method='uniform')
    lbp_region = lbp[region_mask == 1]

    # Histogram of LBP values (10 bins for uniform LBP with P=8)
    lbp_hist, _ = np.histogram(lbp_region, bins=10, range=(0, 10), density=True)
    for i, val in enumerate(lbp_hist):
        features[f'lbp_{i}'] = float(val)

    # --- Edge density ---
    # Sobel edge map, then count edges within region
    edge_map   = sobel(img_uint8.astype(float))
    edge_binary = (edge_map > edge_map.mean()).astype(np.uint8)
    region_size = region_mask.sum()
    features['edge_density'] = float(edge_binary[region_mask == 1].sum() / region_size)

    return features

In [5]:
def extract_region_features(image, region_mask):
    """
    Given a normalized image and a binary region mask,
    extracts all handcrafted texture features for that region.
    Returns a flat dictionary of features.
    """
    features = {}

    # Get pixel values in this region
    pixels = image[region_mask == 1]

    # Skip empty regions
    if len(pixels) < 10:
        return None

    # --- Basic intensity stats ---
    features['mean_intensity'] = float(np.mean(pixels))
    features['std_intensity']  = float(np.std(pixels))
    features['skewness']       = float(skew(pixels))
    features['kurtosis']       = float(kurtosis(pixels))

    # --- Entropy ---
    # need histogram of pixel values for entropy
    hist, _ = np.histogram(pixels, bins=256, density=True)
    hist    = hist[hist > 0]   # remove zeros before log
    features['entropy'] = float(entropy(hist))

    # --- GLCM features ---
    # GLCM needs uint8 image
    # clip normalized image back to 0-255 uint8 for GLCM
    img_clipped = np.clip(image, image.min(), image.max())
    img_uint8   = ((img_clipped - img_clipped.min()) /
                   (img_clipped.max() - img_clipped.min()) * 255).astype(np.uint8)

    # Apply mask — set non-region pixels to 0
    img_masked = img_uint8.copy()
    img_masked[region_mask == 0] = 0

    # Reduce to 64 gray levels for GLCM (faster, less sparse)
    img_64 = (img_masked // 4).astype(np.uint8)

    glcm = graycomatrix(
        img_64,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=64,
        symmetric=True,
        normed=True
    )

    # Average across all angles
    features['glcm_contrast']    = float(graycoprops(glcm, 'contrast').mean())
    features['glcm_homogeneity'] = float(graycoprops(glcm, 'homogeneity').mean())
    features['glcm_energy']      = float(graycoprops(glcm, 'energy').mean())
    features['glcm_correlation'] = float(graycoprops(glcm, 'correlation').mean())

    # --- LBP histogram features ---
    # LBP on full image, then extract values within region
    lbp = local_binary_pattern(img_uint8, P=8, R=1, method='uniform')
    lbp_region = lbp[region_mask == 1]

    # Histogram of LBP values (10 bins for uniform LBP with P=8)
    lbp_hist, _ = np.histogram(lbp_region, bins=10, range=(0, 10), density=True)
    for i, val in enumerate(lbp_hist):
        features[f'lbp_{i}'] = float(val)

    # --- Edge density ---
    # Sobel edge map, then count edges within region
    edge_map   = sobel(img_uint8.astype(float))
    edge_binary = (edge_map > edge_map.mean()).astype(np.uint8)
    region_size = region_mask.sum()
    features['edge_density'] = float(edge_binary[region_mask == 1].sum() / region_size)

    return features

In [6]:
def compute_case_level_features(image, masks):
    """
    Computes features that need values from multiple regions:
    - shadow index = posterior mean / tumor mean
    - lesion to background contrast = tumor mean - background mean
    """
    tumor_pixels    = image[masks['tumor_mask'] == 1]
    posterior_pixels = image[masks['posterior_mask'] == 1]
    background_pixels = image[masks['background_mask'] == 1]

    tumor_mean    = float(np.mean(tumor_pixels))    if len(tumor_pixels) > 0    else 0
    posterior_mean = float(np.mean(posterior_pixels)) if len(posterior_pixels) > 0 else 0
    background_mean = float(np.mean(background_pixels)) if len(background_pixels) > 0 else 0

    shadow_index             = posterior_mean / tumor_mean if tumor_mean != 0 else 0
    lesion_background_contrast = tumor_mean - background_mean

    return {
        'shadow_index'              : shadow_index,
        'lesion_background_contrast': lesion_background_contrast,
    }

In [7]:
# Load first case
sample_row  = manifest.iloc[0]
sample      = np.load(sample_row['npz_path'])

image = sample['image']
masks = {
    'tumor_mask'     : sample['tumor_mask'],
    'ring_5px'       : sample['ring_5px'],
    'ring_15px'      : sample['ring_15px'],
    'posterior_mask' : sample['posterior_mask'],
    'skin_mask'      : sample['skin_mask'],
    'background_mask': sample['background_mask'],
}

# Test one region
feats = extract_region_features(image, masks['tumor_mask'])
print(f"Case: {sample_row['case_id']} | Label: {sample_row['label']}")
print(f"Number of features extracted: {len(feats)}")
print(feats)

Case: benign (1) | Label: benign
Number of features extracted: 20
{'mean_intensity': -0.9997890591621399, 'std_intensity': 0.6228088140487671, 'skewness': 1.6401866674423218, 'kurtosis': 2.0933189392089844, 'entropy': 4.227162745703527, 'glcm_contrast': 0.7334742947424098, 'glcm_homogeneity': 0.9969171037399065, 'glcm_energy': 0.99586731247374, 'glcm_correlation': 0.7353457571107619, 'lbp_0': 0.016260162601626018, 'lbp_1': 0.04065040650406504, 'lbp_2': 0.032520325203252036, 'lbp_3': 0.08943089430894309, 'lbp_4': 0.3780487804878049, 'lbp_5': 0.18699186991869918, 'lbp_6': 0.08943089430894309, 'lbp_7': 0.07317073170731707, 'lbp_8': 0.052845528455284556, 'lbp_9': 0.04065040650406504, 'edge_density': 0.5934959349593496}


In [11]:
REGIONS = {
    'tumor'       : 'tumor_mask',
    'ring_5px'    : 'ring_5px',
    'ring_15px'   : 'ring_15px',
    'posterior'   : 'posterior_mask',
    'skin'        : 'skin_mask',
    'background'  : 'background_mask',
}

all_records = []

for idx, row in manifest.iterrows():
    try:
        sample = np.load(row['npz_path'])
        image  = sample['image']

        masks = {
            'tumor_mask'     : sample['tumor_mask'],
            'ring_5px'       : sample['ring_5px'],
            'ring_15px'      : sample['ring_15px'],
            'posterior_mask' : sample['posterior_mask'],
            'skin_mask'      : sample['skin_mask'],
            'background_mask': sample['background_mask'],
        }

        # Case-level features (shadow index, lesion-background contrast)
        case_feats = compute_case_level_features(image, masks)

        # Per-region features
        for region_name, mask_key in REGIONS.items():
            feats = extract_region_features(image, masks[mask_key])

            if feats is None:
                print(f"  Skipping empty region '{region_name}' for {row['case_id']}")
                continue

            record = {
                'case_id'       : row['case_id'],
                'label'         : row['label'],
                'region'        : region_name,
            }
            record.update(feats)

            # Only add case-level features to tumor row (avoids duplication)
            if region_name == 'tumor':
                record.update(case_feats)

            all_records.append(record)

    except Exception as e:
        print(f"ERROR on {row['case_id']}: {e}")

print(f"\nDone! Total rows: {len(all_records)}")
print(f"Expected (~6 regions × {len(manifest)} cases): {6 * len(manifest)}")

  Skipping empty region 'tumor' for normal (1)
  Skipping empty region 'ring_5px' for normal (1)
  Skipping empty region 'ring_15px' for normal (1)
  Skipping empty region 'posterior' for normal (1)
  Skipping empty region 'tumor' for normal (10)
  Skipping empty region 'ring_5px' for normal (10)
  Skipping empty region 'ring_15px' for normal (10)
  Skipping empty region 'posterior' for normal (10)
  Skipping empty region 'tumor' for normal (100)
  Skipping empty region 'ring_5px' for normal (100)
  Skipping empty region 'ring_15px' for normal (100)
  Skipping empty region 'posterior' for normal (100)
  Skipping empty region 'tumor' for normal (101)
  Skipping empty region 'ring_5px' for normal (101)
  Skipping empty region 'ring_15px' for normal (101)
  Skipping empty region 'posterior' for normal (101)
  Skipping empty region 'tumor' for normal (102)
  Skipping empty region 'ring_5px' for normal (102)
  Skipping empty region 'ring_15px' for normal (102)
  Skipping empty region 'poste

In [12]:
df_features = pd.DataFrame(all_records)

# Fill NaN for shadow_index and lesion_background_contrast
# (only tumor rows have these — fill others with 0)
df_features['shadow_index']               = df_features['shadow_index'].fillna(0)
df_features['lesion_background_contrast'] = df_features['lesion_background_contrast'].fillna(0)

# Save
OUTPUT_CSV = DATASET_PATH / 'features_texture.csv'
df_features.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
print(f"Shape: {df_features.shape}")
df_features.head(12)

Saved: C:\Users\jessi\OneDrive\Desktop\aiims2\features_texture.csv
Shape: (4148, 25)


,case_id,label,region,mean_intensity,std_intensity,skewness,kurtosis,entropy,glcm_contrast,glcm_homogeneity,...,lbp_3,lbp_4,lbp_5,lbp_6,lbp_7,lbp_8,lbp_9,edge_density,shadow_index,lesion_background_contrast
0,benign (1),benign,tumor,-0.999789,0.622809,1.640187,2.093319,4.227163,0.733474,0.996917,...,0.089431,0.378049,0.186992,0.089431,0.073171,0.052846,0.040650,0.593496,0.264460,-0.983460
1,benign (1),benign,ring_5px,0.900587,0.733801,-0.322381,0.811008,4.536022,3.620686,0.995492,...,0.144615,0.332308,0.169231,0.043077,0.052308,0.036923,0.064615,0.584615,0.000000,0.000000
2,benign (1),benign,ring_15px,1.051122,0.502193,0.752453,-0.601233,4.448174,5.911933,0.989186,...,0.118947,0.261053,0.134737,0.081053,0.080000,0.054737,0.093684,0.390526,0.000000,0.000000
3,benign (1),benign,posterior,-0.264404,1.070727,0.213826,-1.015387,5.366711,17.295755,0.626801,...,0.129345,0.247249,0.140509,0.082401,0.078749,0.060281,0.088989,0.310905,0.000000,0.000000
4,benign (1),benign,skin,0.470129,0.635451,0.423734,-0.044272,4.925203,6.688780,0.921917,...,0.110026,0.288021,0.139844,0.065625,0.082812,0.051693,0.086198,0.458984,0.000000,0.000000
5,benign (1),benign,background,-0.016329,0.995576,-0.214900,-0.811224,5.332696,19.047253,0.427882,...,0.126970,0.245333,0.140576,0.082340,0.079684,0.058611,0.089088,0.331719,0.000000,0.000000
6,benign (10),benign,tumor,-1.609278,0.349434,3.722780,18.730455,2.541511,1.568532,0.980227,...,0.069737,0.140770,0.073622,0.038661,0.021828,0.529412,0.048465,0.150388,0.067225,-1.706996
7,benign (10),benign,ring_5px,0.504072,0.952880,0.470433,-0.422797,5.195233,8.293234,0.981451,...,0.152767,0.448168,0.153546,0.053780,0.018706,0.013250,0.042868,0.763055,0.000000,0.000000
8,benign (10),benign,ring_15px,0.959165,0.947055,0.454106,-0.629433,4.833711,12.190492,0.972970,...,0.147400,0.250281,0.170595,0.078563,0.068837,0.088290,0.063599,0.395810,0.000000,0.000000
9,benign (10),benign,posterior,-0.108183,0.974311,0.830108,0.123762,5.286333,9.189247,0.696456,...,0.136691,0.258845,0.147266,0.084487,0.072935,0.059431,0.076507,0.320480,0.000000,0.000000


In [10]:
print("=== Rows per region ===")
print(df_features['region'].value_counts())

print("\n=== Rows per label ===")
print(df_features['label'].value_counts())

print("\n=== Sample of tumor region features ===")
tumor_df = df_features[df_features['region'] == 'tumor']
print(tumor_df[['case_id', 'label', 'mean_intensity', 'std_intensity',
                 'entropy', 'glcm_contrast', 'shadow_index',
                 'lesion_background_contrast']].head(8))

=== Rows per region ===
region
skin          780
background    779
ring_5px      647
tumor         647
posterior     647
ring_15px     647
Name: count, dtype: int64

=== Rows per label ===
label
benign       2621
malignant    1260
normal        266
Name: count, dtype: int64

=== Sample of tumor region features ===
         case_id   label  mean_intensity  std_intensity   entropy  \
0     benign (1)  benign       -0.999789       0.622809  4.227163   
6    benign (10)  benign       -1.609278       0.349434  2.541511   
12  benign (100)  benign       -0.293976       0.550632  4.620606   
18  benign (101)  benign       -0.573224       0.542288  4.402359   
24  benign (102)  benign       -1.083037       0.439115  3.499399   
30  benign (103)  benign       -1.381518       0.308837  3.409517   
36  benign (104)  benign       -0.412178       0.730499  4.669758   
42  benign (105)  benign       -1.153062       0.614055  4.391370   

    glcm_contrast  shadow_index  lesion_background_contrast  
